In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
SYSTEM_PROMPT = """
## Role

You are a **Spark SQL tool-using agent** responsible for answering data-related questions by generating and executing SQL queries on a Databricks Lakehouse environment.  
You interact with a MCP server which provides tools that can be used to analyze and query Databricks.

You MUST use the provided tools to obtain all information — never rely on your own assumptions or memory.  
Do **not** respond conversationally or with confirmations like "Got it".  
Every single response must either:
1. Call one or more tools to gather information or execute queries, OR
2. Return a final output containing both the executed SQL query and its markdown-formatted results.

---

## Environment Context

- The Spark session is connected to the Databricks Lakehouse using **Databricks Connect**.
- Use the following catalog and schema names:
  - **Catalog:** `{CATALOG_NAME}`
  - **Schema:** `{SCHEMA_NAME}`
- All queries must explicitly reference this context in the form:

```
SELECT * FROM <catalog>.<schema>.<table_name>
```

- Always use Spark SQL dialect conventions (Databricks SQL), including functions, syntax, and operators supported by Spark 3.x+.

## Tool Usage Policy

For **every user query**:
1. Start by listing all tables in <catalog>.<schema> using appropriate tool to see what tables exist. If that is all the user asked then return these results.
2. Then fetch the schema for any relevant tables to understand structure and columns.  
3. Use that schema information to construct a **fully qualified** Spark SQL query referencing the correct catalog and schema.  
4. Validate the query so that your are limiting the number of rows returned and for any syntax error or query optimizations.  
5. Only after validating the query execute it and return the results.

If any tool returns an error, summarize it clearly.  
Do **not** include full stack traces — only the main error message and a concise explanation.

## Critical Reminders
- Always use **fully qualified table names**: `<catalog>.<schema>.<table>`.
- Use **Spark SQL syntax only** (no T-SQL, MySQL, or Postgres syntax).
- Do not invent column names, table names, or joins.
- Only base your queries on information gathered from the tools.
- Return concise, structured, markdown-formatted outputs.
"""

In [3]:
from mcp.client.streamable_http import streamablehttp_client
from langchain_mcp_adapters.prompts import load_mcp_prompt
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.chat_models import ChatOllama
from langgraph.prebuilt import create_react_agent
from mcp import ClientSession


CATALOG = os.environ.get("UC_CATALOG_NAME", "tpch")
SCHEMA = os.environ.get("UC_SCHEMA_NAME", "bronze")

DATABRICKS_MCP_HOST = os.environ.get("DATABRICKS_MCP_HOST")

async def run_agent_once(messages: str):
    async with streamablehttp_client(f"{DATABRICKS_MCP_HOST}/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            spark_sql_agent_llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0.0)
            spark_sql_agent_prompt = ChatPromptTemplate([
                    ("system", SYSTEM_PROMPT.format(**{"CATALOG_NAME": CATALOG, "SCHEMA_NAME": SCHEMA})),
                    ("placeholder", "{messages}"),
                    ("placeholder", "{agent_scratchpad}"),
            ])
            await session.initialize()
            tools = await load_mcp_tools(session, server_name="databricks_mcp_server")
            agent = create_react_agent(spark_sql_agent_llm, tools=tools, prompt=spark_sql_agent_prompt)
            return await agent.ainvoke({"messages": messages})
        
response = await run_agent_once("List all tools available for you.")
response

{'messages': [HumanMessage(content='List all tools available for you.', additional_kwargs={}, response_metadata={}, id='84994eef-059d-4043-a319-1bcd01d16d84'),
  AIMessage(content='**Available tools**\n\n| Tool | Description |\n|------|-------------|\n| `fetch_schemas_in_catalog` | Retrieves all schemas within a given Unity Catalog catalog. |\n| `fetch_tables_in_schema` | Retrieves all tables within a given Unity Catalog catalog and schema. |\n| `fetch_table_info` | Provides a detailed description (columns, data types, lineage) for one or more Unity Catalog tables. |\n| `execute_spark_sql_query` | Executes a read‑only Spark SQL `SELECT` query against the Databricks warehouse and returns the results. |', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b-cloud', 'created_at': '2025-11-21T16:35:43.168635172Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2307594857, 'load_duration': None, 'prompt_eval_count': 1376, 'prompt_eval_duration': None, 'eval_count': 236, '

In [4]:
print(response["messages"][-1].content)

**Available tools**

| Tool | Description |
|------|-------------|
| `fetch_schemas_in_catalog` | Retrieves all schemas within a given Unity Catalog catalog. |
| `fetch_tables_in_schema` | Retrieves all tables within a given Unity Catalog catalog and schema. |
| `fetch_table_info` | Provides a detailed description (columns, data types, lineage) for one or more Unity Catalog tables. |
| `execute_spark_sql_query` | Executes a read‑only Spark SQL `SELECT` query against the Databricks warehouse and returns the results. |


In [5]:
response = await run_agent_once("List the top 3 nations based on the total number of customers from that nation.")
response

{'messages': [HumanMessage(content='List the top 3 nations based on the total number of customers from that nation.', additional_kwargs={}, response_metadata={}, id='bc53fe08-6e78-44e2-8c42-dd8828daf663'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b-cloud', 'created_at': '2025-11-21T16:35:44.598165866Z', 'done': True, 'done_reason': 'stop', 'total_duration': 750963439, 'load_duration': None, 'prompt_eval_count': 1386, 'prompt_eval_duration': None, 'eval_count': 72, 'eval_duration': None, 'model_name': 'gpt-oss:120b-cloud'}, id='run--ac3a36fd-97ca-424f-ad6d-468a64db680a-0', tool_calls=[{'name': 'fetch_tables_in_schema', 'args': {'catalog': 'tpch', 'schema': 'bronze'}, 'id': '495a7d5d-880b-473b-bd88-166bea46762d', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1386, 'output_tokens': 72, 'total_tokens': 1458}),
  ToolMessage(content='# List of tables in `tpch.bronze`\n\n*Number of tables*: 8\n\n## Table names:\n\ncustomer, lineitem, nati

In [7]:
for msg in response["messages"]:
    print(msg.content)

List the top 3 nations based on the total number of customers from that nation.

# List of tables in `tpch.bronze`

*Number of tables*: 8

## Table names:

customer, lineitem, nation, orders, part, partsupp, region, supplier


# Table Information

## Table
**Name:** `tpch.bronze.customer`
**Type:** `MANAGED`
**Data Format:** `DELTA`
**Description:** No description

### Schema
| Column | Type | Nullable | Comment |
|--------|------|----------|---------|
| c_custkey | bigint | TRUE |  |
| c_name | string | TRUE |  |
| c_address | string | TRUE |  |
| c_nationkey | bigint | TRUE |  |
| c_phone | string | TRUE |  |
| c_acctbal | decimal(18,2) | TRUE |  |
| c_mktsegment | string | TRUE |  |
| c_comment | string | TRUE |  |

### Constraints
No constraints

---

## Table
**Name:** `tpch.bronze.nation`
**Type:** `MANAGED`
**Data Format:** `DELTA`
**Description:** No description

### Schema
| Column | Type | Nullable | Comment |
|--------|------|----------|---------|
| n_nationkey | bigint | TR